# First-Order Equations

A first-order ODE $y' = f(t, y)$ gives the slope of the solution at every point. Together with a starting value $y(t_0) = y_0$ it fixes the whole path. This notebook draws slope fields, solves classic equations exactly and numerically, and writes Euler's method by hand.

In [ ]:
Pollis.packages("OrdinaryDiffEq", "Plots")

In [ ]:
using OrdinaryDiffEq, Plots

## Slope fields

Logistic growth $y' = r y (1 - y/K)$: small populations grow at rate $r$, and growth stops at the carrying capacity $K$. Each short segment shows the slope $f(t, y)$; solutions follow the segments.

In [ ]:
r, K = 0.8, 10.0
f(y, p, t) = r * y * (1 - y / K)
ts = range(0, 10, length=21); ys = range(0, 14, length=15)
T = vec([t for t in ts, y in ys]); Y = vec([y for t in ts, y in ys])
S = [f(y, nothing, t) for (t, y) in zip(T, Y)]
len = @. sqrt(1 + (S * step(ts) / step(ys))^2)   # segment length in grid units, so all segments look equal
quiver(T, Y, quiver=(0.4 .* step(ts) ./ len, 0.4 .* S .* step(ts) ./ len), c=:gray, label="", xlabel="t", ylabel="y", title="Slope field of logistic growth")
for y0 in (0.5, 2.0, 13.0)
    plot!(solve(ODEProblem(f, y0, (0.0, 10.0)), Tsit5()), lw=2, label="y0 = $y0")
end
current()

## Checking against the exact solution

Separating variables gives $y(t) = K / \left(1 + (K/y_0 - 1)e^{-rt}\right)$. With a tight tolerance the numerical solution agrees to about ten digits, and `sol(t)` can be evaluated at any time, not just at the solver's steps.

In [ ]:
logistic(t; y0=0.5) = K / (1 + (K / y0 - 1) * exp(-r * t))
sol = solve(ODEProblem(f, 0.5, (0.0, 10.0)), Tsit5(); reltol=1e-10, abstol=1e-12)
(largest_error = maximum(abs(sol(t) - logistic(t)) for t in 0:0.01:10), steps = length(sol.t) - 1)

## Linear equations: cooling and price adjustment

**Newton's law of cooling**, $T' = -k(T - T_a)$: a coffee at 90 degrees in a 20-degree room. The solution $T_a + (T_0 - T_a)e^{-kt}$ approaches room temperature exponentially.

In [ ]:
k, Ta = 0.1, 20.0
cool(T, p, t) = -k * (T - Ta)
sol = solve(ODEProblem(cool, 90.0, (0.0, 30.0)), Tsit5())
plot(sol, lw=2, label="numerical", xlabel="minutes", ylabel="temperature")
scatter!(0:2:30, t -> Ta + (90 - Ta) * exp(-k * t), ms=3, label="exact")

In [ ]:
(minutes_to_reach_60 = log((90 - Ta) / (60 - Ta)) / k,)

**Walrasian price adjustment**: the price rises when demand exceeds supply, $p' = \kappa\,(D(p) - S(p))$. With linear demand $a - bp$ and supply $c + dp$ the equation is linear. The price converges to $p^* = (a - c)/(b + d)$ at rate $\kappa(b + d)$.

In [ ]:
a, b, c, d, kappa = 100.0, 2.0, 10.0, 1.0, 0.5
excess(p) = (a - b * p) - (c + d * p)
price(p, _, t) = kappa * excess(p)
pstar = (a - c) / (b + d)
sol = solve(ODEProblem(price, 10.0, (0.0, 5.0)), Tsit5())
plot(sol, lw=2, label="price", xlabel="t"); hline!([pstar], ls=:dash, label="equilibrium price")

In [ ]:
(equilibrium_price = pstar, half_life = log(2) / (kappa * (b + d)), price_at_t_1 = sol(1.0))

## Euler's method by hand

Euler's method follows the slope for a step $h$: $y_{n+1} = y_n + h f(t_n, y_n)$. Its error at a fixed time is proportional to $h$, so halving the step halves the error. The ratios in the table approach 2.

In [ ]:
function euler(f, y0, (t0, t1), n)
    h = (t1 - t0) / n
    t, y = t0, y0
    for _ in 1:n
        y += h * f(y, nothing, t)
        t += h
    end
    return y
end
errs = [abs(euler(f, 0.5, (0.0, 10.0), n) - logistic(10.0)) for n in (10, 20, 40, 80, 160, 320)]
[(n = 10 * 2^(i - 1), error = errs[i], ratio = i == 1 ? missing : errs[i - 1] / errs[i]) for i in eachindex(errs)]

## Equilibria and the phase line

For an autonomous equation $y' = f(y)$, equilibria are the zeros of $f$. An equilibrium is stable when $f'(y^*) < 0$ (above it $y$ falls, below it $y$ rises), unstable when $f'(y^*) > 0$. For logistic growth, $y = 0$ is unstable and $y = K$ stable.

In [ ]:
yy = range(-1, 12, length=300)
plot(yy, [f(y, nothing, 0.0) for y in yy], lw=2, label="f(y)", xlabel="y", ylabel="dy/dt", title="Phase line: y rises where f > 0, falls where f < 0")
hline!([0], c=:gray, label="")
scatter!([0, K], [0, 0], ms=6, label="equilibria")

## Exercises

1. Change the logistic starting value to 15. What does the slope field predict, and what does the solver give?
2. The harvested logistic $y' = ry(1 - y/K) - H$ has two equilibria when $H < rK/4$. Find them for $H = 1$ and decide which is stable. What happens for $H = 2.5$?
3. How many Euler steps are needed for an error below $10^{-4}$ at $t = 10$? Compare with the number of steps Tsit5 took above.
4. Solve $y' = y^2$, $y(0) = 1$ on $(0, 2)$. Look at `sol.retcode` and `sol.t[end]` and explain what happened.